# 03 — Gold referral model

Build board-reporting views from conformed Silver tables. Cross-table lifecycle
dates and durations belong here because they depend on referrals, provider
responses, offers, IPAs, and events.

`PlacementUrgencyBand` below is an **operational target interval**, derived from
created date to required placement date. It is not a safeguarding assessment and
must not be interpreted as inferred child criticality.

In [ ]:
GOLD_SCHEMA = "gold"
SNAPSHOT_TABLE = "gold.fact_referral_snapshot"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS gold.cfg_placement_urgency_rule (
  PlacementUrgencyBand STRING,
  MaximumTargetDays INT,
  WarningHoursBeforeTarget INT,
  SortOrder INT,
  IsActive BOOLEAN
) USING DELTA;

MERGE INTO gold.cfg_placement_urgency_rule AS t
USING (
  SELECT * FROM VALUES
    ('Critical', 1, 6, 1, true),
    ('High', 3, 24, 2, true),
    ('Medium', 7, 48, 3, true),
    ('Planned', 99999, 72, 4, true)
  AS v(PlacementUrgencyBand, MaximumTargetDays, WarningHoursBeforeTarget, SortOrder, IsActive)
) AS s
ON t.PlacementUrgencyBand = s.PlacementUrgencyBand
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [ ]:
%%sql
CREATE OR REPLACE VIEW gold.fact_referral AS
WITH offer_rollup AS (
  SELECT
    rp.referral_id,
    MIN(o.offer_date) AS FirstOfferDate,
    MIN(CASE WHEN lower(o.offer_status) IN ('accepted','approved','selected') THEN o.last_modified_date END) AS OfferAcceptedDate,
    COUNT(DISTINCT o.offer_id) AS OfferCount,
    COUNT(DISTINCT o.provider_home_id) AS UniqueHomesOffered,
    MAX(o.last_modified_date) AS LastOfferActivityDate
  FROM silver.slv_offer_offer o
  INNER JOIN silver.slv_referral_referral_provider rp
    ON o.referral_provider_id = rp.referral_provider_id
  GROUP BY rp.referral_id
),
ipa_rollup AS (
  SELECT
    referral_id,
    MIN(created_datetime) AS IPAIssuedDate,
    MIN(placement_admission_date) AS PlannedPlacementStartDate,
    SUM(costs_total_weekly_fee) AS EstimatedWeeklyCost,
    MAX(updated_datetime) AS LastIPAActivityDate
  FROM silver.slv_ipa_ipa
  GROUP BY referral_id
),
event_rollup AS (
  SELECT referral_id,
    MIN(event_timestamp) AS FirstActionDate,
    MAX(event_timestamp) AS LastEventActivityDate
  FROM silver.slv_referral_referral_event_log
  GROUP BY referral_id
)
SELECT
  r.referral_id AS ReferralID,
  r.created_timestamp AS ReferralCreatedDate,
  r.required_start_date AS RequiredPlacementDate,
  r.response_required_by_date AS ResponseRequiredDate,
  e.FirstActionDate,
  o.FirstOfferDate,
  o.OfferAcceptedDate,
  i.IPAIssuedDate,
  CASE WHEN lower(r.status) IN ('closed','cancelled','withdrawn','completed') THEN r.modified_timestamp END AS ReferralClosedDate,
  CAST(NULL AS STRING) AS ReferralClosureReason,
  greatest(r.modified_timestamp, e.LastEventActivityDate, o.LastOfferActivityDate, i.LastIPAActivityDate) AS LastActivityDate,
  r.status AS CurrentStatus,
  r.placement_type_code AS PlacementTypeRequired,
  CASE
    WHEN r.required_start_date IS NULL THEN 'Unspecified'
    WHEN datediff(r.required_start_date, to_date(r.created_timestamp)) <= 1 THEN 'Critical'
    WHEN datediff(r.required_start_date, to_date(r.created_timestamp)) <= 3 THEN 'High'
    WHEN datediff(r.required_start_date, to_date(r.created_timestamp)) <= 7 THEN 'Medium'
    ELSE 'Planned'
  END AS PlacementUrgencyBand,
  CAST(NULL AS STRING) AS ChildCriticalityCode,
  coalesce(o.OfferCount, 0) AS OfferCount,
  coalesce(o.UniqueHomesOffered, 0) AS UniqueHomesOffered,
  CASE WHEN coalesce(o.OfferCount, 0) > 0 THEN true ELSE false END AS HasOffer,
  datediff(to_date(e.FirstActionDate), to_date(r.created_timestamp)) AS DaysToFirstAction,
  datediff(to_date(o.FirstOfferDate), to_date(r.created_timestamp)) AS DaysToFirstOffer,
  datediff(to_date(o.OfferAcceptedDate), to_date(r.created_timestamp)) AS DaysToAcceptedOffer,
  datediff(to_date(i.IPAIssuedDate), to_date(r.created_timestamp)) AS DaysToIPA,
  datediff(coalesce(to_date(CASE WHEN lower(r.status) IN ('closed','cancelled','withdrawn','completed') THEN r.modified_timestamp END), current_date()), to_date(r.created_timestamp)) AS DaysOpen,
  datediff(current_date(), to_date(greatest(r.modified_timestamp, e.LastEventActivityDate, o.LastOfferActivityDate, i.LastIPAActivityDate))) AS DaysWithoutActivity,
  greatest(datediff(current_date(), r.required_start_date), 0) AS DaysPastRequiredDate,
  CASE WHEN lower(r.status) NOT IN ('closed','cancelled','withdrawn','completed') THEN true ELSE false END AS IsOpen,
  CASE WHEN i.IPAIssuedDate IS NOT NULL AND to_date(i.IPAIssuedDate) <= r.required_start_date THEN true ELSE false END AS PlacedByRequiredDate,
  CASE
    WHEN i.IPAIssuedDate IS NOT NULL AND to_date(i.IPAIssuedDate) <= r.required_start_date THEN 'Placed by target'
    WHEN i.IPAIssuedDate IS NOT NULL THEN 'Placed after target'
    WHEN r.required_start_date < current_date() AND lower(r.status) NOT IN ('closed','cancelled','withdrawn','completed') THEN 'Open overdue'
    WHEN lower(r.status) NOT IN ('closed','cancelled','withdrawn','completed') THEN 'Open on track'
    ELSE 'Closed without placement'
  END AS RequiredPlacementDateOutcome,
  i.PlannedPlacementStartDate,
  i.EstimatedWeeklyCost,
  current_timestamp() AS GoldModelledAt
FROM silver.slv_referral_referral r
LEFT JOIN offer_rollup o ON r.referral_id = o.referral_id
LEFT JOIN ipa_rollup i ON r.referral_id = i.referral_id
LEFT JOIN event_rollup e ON r.referral_id = e.referral_id;

In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

snapshot = spark.table("gold.fact_referral").select(
    F.current_date().alias("SnapshotDate"),
    "ReferralID", "CurrentStatus", "PlacementUrgencyBand", "RequiredPlacementDate",
    "IsOpen", "HasOffer", "OfferCount", "DaysOpen", "DaysWithoutActivity",
    "DaysPastRequiredDate", "PlacedByRequiredDate", "RequiredPlacementDateOutcome"
)

if not spark.catalog.tableExists(SNAPSHOT_TABLE):
    snapshot.write.format("delta").mode("overwrite").saveAsTable(SNAPSHOT_TABLE)
else:
    target = DeltaTable.forName(spark, SNAPSHOT_TABLE)
    (target.alias("t").merge(snapshot.alias("s"),
        "t.SnapshotDate = s.SnapshotDate AND t.ReferralID = s.ReferralID")
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())

print(f"Snapshot refreshed: {snapshot.count():,} referrals")

In [ ]:
%%sql
CREATE OR REPLACE VIEW gold.vw_provider_offer_performance AS
SELECT
  rp.provider_id AS ProviderID,
  COUNT(DISTINCT rp.referral_id) AS ReferralsReceived,
  COUNT(DISTINCT o.offer_id) AS OffersSubmitted,
  COUNT(DISTINCT CASE WHEN lower(o.offer_status) IN ('accepted','approved','selected') THEN o.offer_id END) AS OffersAccepted,
  COUNT(DISTINCT CASE WHEN i.created_datetime IS NOT NULL AND to_date(i.created_datetime) <= r.required_start_date THEN r.referral_id END) AS ReferralsPlacedByTarget
FROM silver.slv_referral_referral_provider rp
LEFT JOIN silver.slv_referral_referral r ON rp.referral_id = r.referral_id
LEFT JOIN silver.slv_offer_offer o ON rp.referral_provider_id = o.referral_provider_id
LEFT JOIN silver.slv_ipa_ipa i ON r.referral_id = i.referral_id
GROUP BY rp.provider_id;